In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
from google.colab import drive

"""
@function initialize_step_1_workspace
@description Creates a specialized directory structure for the Mirpur DOHS
             Digital Twin construction phase.
"""
def initialize_step_1_workspace():
    # Primary directory for the first phase of research
    base_step_path = '/content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin'

    sub_folders = [
        '3d_models',   # For .glb and .obj exports for Blender
        'metadata',    # For building heights and coordinate CSVs (Table 1)
        'renders',     # For high-res PNGs for the manuscript (Fig 1)
        'logs'         # For tracking OpenStreetMap fetch success
    ]

    print(f"📂 Initializing Step 1: Digital Twin Workspace...")

    for folder in sub_folders:
        full_path = os.path.join(base_step_path, folder)
        if not os.path.exists(full_path):
            os.makedirs(full_path)
            print(f"✅ Created: {full_path}")
        else:
            print(f"ℹ️  Exists: {full_path}")

    return base_step_path

# Execute workspace setup
STEP_1_PATH = initialize_step_1_workspace()

📂 Initializing Step 1: Digital Twin Workspace...
ℹ️  Exists: /content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin/3d_models
ℹ️  Exists: /content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin/metadata
ℹ️  Exists: /content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin/renders
ℹ️  Exists: /content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin/logs


2. Digital Twin Construction (Mirpur DOHS v1)
Now, we run the Real-World Reconstruction. This code fetches the actual Mirpur DOHS geometry and saves it directly into your new folders.

File: Colab Notebook (Account A): /content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin/generate_scene.py

In [3]:
# Fix for NumPy 2.0 compatibility issues
!pip install "numpy<2.0" osmnx sionna
import os
import osmnx as ox
import pandas as pd
import numpy as np
import tensorflow as tf
import sionna
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera

# Safety Check: Ensure STEP_1_PATH is defined
try:
    STEP_1_PATH
except NameError:
    print("⚠️  STEP_1_PATH not found. Re-initializing workspace...")
    STEP_1_PATH = '/content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin'
    for folder in ['3d_models', 'metadata', 'renders', 'logs']:
        os.makedirs(os.path.join(STEP_1_PATH, folder), exist_ok=True)

def build_mirpur_dohs_manuscript_v1(dir_path, lat=23.8365, lon=90.3695, radius=400):
    print("🛰️  Fetching real Mirpur DOHS geometry from OpenStreetMap...")
    buildings = ox.features_from_point((lat, lon), tags={'building': True}, dist=radius)

    # 1. XML Scene Header - Fixed Material Name to start with itu_
    xml_content = f"""<scene version=\"2.1.0\">
    <integrator type=\"path\"/>
    <bsdf type=\"itu_radio_material\" id=\"itu_concrete\">
        <string name=\"material\" value=\"concrete\"/>
    </bsdf>
    """

    metadata = []
    print(f"🏗️  Processing {len(buildings)} real structures...")

    for idx, row in buildings.iterrows():
        if row.geometry.geom_type != 'Polygon': continue

        centroid = row.geometry.centroid
        x = float((centroid.x - lon) * 111000 * np.cos(np.radians(lat)))
        y = float((centroid.y - lat) * 111000)

        bounds = row.geometry.bounds # [minx, miny, maxx, maxy]
        w = float((bounds[2] - bounds[0]) * 111000 * np.cos(np.radians(lat)))
        d = float((bounds[3] - bounds[1]) * 111000)

        # Mirpur DOHS specific heights
        levels = float(row['building:levels']) if 'building:levels' in row and not np.isnan(float(row['building:levels'])) else np.random.uniform(6, 11)
        h = levels * 3.2

        # 2. Add Building to XML
        xml_content += f"""
    <shape type=\"cube\" id=\"Bldg_{idx}\">
        <transform name=\"to_world\">
            <scale x=\"{w/2}\" y=\"{d/2}\" z=\"{h/2}\"/>
            <translate x=\"{x}\" y=\"{y}\" z=\"{h/2}\"/>
        </transform>
        <ref id=\"itu_concrete\"/>
    </shape>"""

        metadata.append({"ID": f"Bldg_{idx}", "Lat": centroid.y, "Lon": centroid.x, "Height": h})

    xml_content += "\n</scene>"

    # 3. Save Files
    models_dir = os.path.join(dir_path, "3d_models")
    os.makedirs(models_dir, exist_ok=True)

    xml_path = os.path.join(models_dir, "mirpur_dohs.xml")
    with open(xml_path, "w") as f:
        f.write(xml_content)

    df_meta = pd.DataFrame(metadata)
    df_meta.to_csv(os.path.join(dir_path, "metadata", "table1_dohs_metadata.csv"), index=False)

    print(f"✅ Digital Twin File Generated: {xml_path}")
    return xml_path, df_meta

# Execute
xml_file, meta_df = build_mirpur_dohs_manuscript_v1(STEP_1_PATH)

# 4. Load Scene
scene = load_scene(xml_file)
scene.frequency = 28e9
print("🔥 Real-world Mirpur DOHS Scene Loaded successfully into Sionna RT.")

🛰️  Fetching real Mirpur DOHS geometry from OpenStreetMap...
🏗️  Processing 1745 real structures...
✅ Digital Twin File Generated: /content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin/3d_models/mirpur_dohs.xml
🔥 Real-world Mirpur DOHS Scene Loaded successfully into Sionna RT.


In [ ]:
import os
import osmnx as ox
import pandas as pd
import numpy as np
import tensorflow as tf
import sionna
import matplotlib.pyplot as plt
from sionna.rt import load_scene, Transmitter, Receiver, Camera
from google.colab import drive

# ==========================================
# 1. Workspace Initialization (Step 1 Folder)
# ==========================================
STEP_1_DIR = '/content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin'
for f in ['metadata', 'renders', 'models']:
    os.makedirs(os.path.join(STEP_1_DIR, f), exist_ok=True)

print(f"📂 Research Workspace Initialized: {STEP_1_DIR}")

# ==========================================
# 2. Digital Twin Reconstruction (Real Map)
# ==========================================
def build_mirpur_dohs_digital_twin_v1_2(dir_path, lat=23.8365, lon=90.3695, radius=400):
    print("🛰️  Fetching live geometry from OpenStreetMap for Mirpur DOHS...")
    buildings = ox.features_from_point((lat, lon), tags={'building': True}, dist=radius)

    xml_content = "<scene version=\"2.1.0\">\n    <integrator type=\"path\"/>\n    <bsdf type=\"itu_radio_material\" id=\"itu_concrete\">\n        <string name=\"material\" value=\"concrete\"/>\n    </bsdf>\n    "

    metadata = []
    print(f"🏗️  Extruding {len(buildings)} real-world structures...")

    for idx, row in buildings.iterrows():
        if row.geometry.geom_type != 'Polygon': continue

        centroid = row.geometry.centroid
        x = float((centroid.x - lon) * 111000 * np.cos(np.radians(lat)))
        y = float((centroid.y - lat) * 111000)

        bounds = row.geometry.bounds
        w = float(bounds[2]-bounds[0])*111000*np.cos(np.radians(lat))
        d = float(bounds[3]-bounds[1])*111000

        levels = float(row['building:levels']) if 'building:levels' in row and not np.isnan(float(row['building:levels'])) else np.random.uniform(6, 11)
        h = levels * 3.2

        xml_content += f"\n    <shape type=\"cube\" id=\"Bldg_{idx}\">\n        <transform name=\"to_world\">\n            <scale x=\"{max(w/2, 1.0)}\" y=\"{max(d/2, 1.0)}\" z=\"{h/2}\"/>\n            <translate x=\"{x}\" y=\"{y}\" z=\"{h/2}\"/>\n        </transform>\n        <ref id=\"itu_concrete\"/>\n    </shape>"

        metadata.append({"Building_ID": f"Bldg_{idx}", "Lat": centroid.y, "Lon": centroid.x, "Height_m": h})

    xml_content += "\n</scene>"

    xml_path = os.path.join(dir_path, "models/mirpur_dohs.xml")
    with open(xml_path, "w") as f: f.write(xml_content)
    pd.DataFrame(metadata).to_csv(os.path.join(dir_path, "metadata/table1_dohs_stats.csv"), index=False)

    return xml_path

xml_file = build_mirpur_dohs_digital_twin_v1_2(STEP_1_DIR)
scene = load_scene(xml_file)
scene.frequency = 28e9

# ==========================================
# 3. Figure Generation (Manuscript Figure 1)
# ==========================================
def generate_figure_1(scene, dir_path):
    print("📸 Rendering Figure 1 (Top-Down Research View)...")

    # Setup Camera
    cam = Camera(position=[0, 0, 800], look_at=[0, 0, 0])
    scene.camera = cam

    # Render - Returns a Figure object in this environment
    fig = scene.render(camera=cam, num_samples=512, resolution=[1280, 720])

    # Save result using savefig since img is a Figure object
    output_path = os.path.join(dir_path, "renders/fig1_mirpur_dohs.png")
    fig.savefig(output_path)
    print(f"🖼️  Manuscript Asset saved to: {output_path}")

generate_figure_1(scene, STEP_1_DIR)

📂 Research Workspace Initialized: /content/drive/MyDrive/nlos_gnn_shared/step_1_digital_twin
🛰️  Fetching live geometry from OpenStreetMap for Mirpur DOHS...
🏗️  Extruding 1745 real-world structures...
📸 Rendering Figure 1 (Top-Down Research View)...


In [ ]:
# The Sionna 2.x Scene object is defined by the XML we generated earlier.
# We can use that existing XML file directly for Blender/Mitsuba imports.
blender_export_path = f"{STEP_1_DIR}/models/mirpur_dohs.xml"

import os
if os.path.exists(blender_export_path):
    print(f"✅ Digital Twin is ready for Blender!")
    print(f"📍 File Path: {blender_export_path}")
    print(f"🔗 Instructions: Use the Mitsuba plugin in Blender to import this XML scene.")
else:
    print(f"⚠️  XML file not found at {blender_export_path}. Please re-run the reconstruction cell.")

The Consolidated Part 2 Engine
Run this entire block in Notebook A. It will create the folders, run the physics loop, and save your 10,000 data points. Note: This will take 15–25 minutes.

File: /content/drive/MyDrive/nlos_gnn_shared/step_2_massive_research/simulation_engine.py

In [ ]:
# File: /content/drive/MyDrive/nlos_gnn_shared/step_2_massive_research/simulation_engine.py
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sionna.rt import Transmitter, Receiver, PathSolver, PlanarArray

# ==========================================
# STEP 1: Workspace Initialization
# ==========================================
STEP_2_DIR = '/content/drive/MyDrive/nlos_gnn_shared/step_2_massive_research'
os.makedirs(os.path.join(STEP_2_DIR, 'raw_data'), exist_ok=True)
os.makedirs(os.path.join(STEP_2_DIR, 'logs'), exist_ok=True)
print(f"☒ Step 2 Workspace Initialized: {STEP_2_DIR}")

# ==========================================
# STEP 2 & 3: The Massive Simulation Engine
# ==========================================
def generate_massive_dataset(scene, save_dir, num_tx=50, rx_per_tx=200):
    print(f"☒ Initializing Data Factory (Target: {num_tx * rx_per_tx} Samples)...")
    master_records = []

    p_solver = PathSolver()

    # THE FIX: Use standard dipole pattern for omnidirectional radiation
    scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="dipole", polarization="V")
    scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="dipole", polarization="V")

    # Boundary constraints for Tx placement
    tx_locations = np.random.uniform(-300, 300, (num_tx, 2))

    for t_idx in range(num_tx):
        # --- MEMORY MANAGEMENT ---
        for tx_name in list(scene.transmitters.keys()): scene.remove(tx_name)
        for rx_name in list(scene.receivers.keys()): scene.remove(rx_name)

        # --- DEVICE PLACEMENT ---
        tx_x = float(tx_locations[t_idx, 0])
        tx_y = float(tx_locations[t_idx, 1])
        scene.add(Transmitter(name=f"Tx_{t_idx}", position=[tx_x, tx_y, 30.0]))

        for r_idx in range(rx_per_tx):
            dist = float(np.random.uniform(20, 350))
            angle = float(np.random.uniform(0, 2*np.pi))

            rx_x = float(tx_x + dist * np.cos(angle))
            rx_y = float(tx_y + dist * np.sin(angle))

            scene.add(Receiver(name=f"Rx_{t_idx}_{r_idx}", position=[rx_x, rx_y, 1.5]))

        # --- PHYSICS SIMULATION ---
        paths = p_solver(scene, max_depth=5, samples_per_src=100000)
        a, _ = paths.cir()

        # --- DATA EXTRACTION & FEATURE ENGINEERING ---
        for r_idx in range(rx_per_tx):
            # Correct indexing for list of tensors
            rx_power = tf.reduce_mean(tf.reduce_sum(tf.square(tf.abs(a[0][r_idx])), axis=-1))
            pl_db = float((-10.0 * tf.math.log(rx_power + 1e-12) / tf.math.log(10.0)).numpy())

            rx_pos = scene.receivers[f"Rx_{t_idx}_{r_idx}"].position.numpy()
            dist_2d = float(np.sqrt((rx_pos[0]-tx_x)**2 + (rx_pos[1]-tx_y)**2))

            master_records.append({
                "scenario_id": t_idx,
                "tx_x": tx_x, "tx_y": tx_y,
                "rx_x": float(rx_pos[0]), "rx_y": float(rx_pos[1]),
                "distance_2d": dist_2d,
                "path_loss_db": pl_db
            })

        if (t_idx + 1) % 5 == 0:
            print(f"✅ Processed Base Station {t_idx + 1}/{num_tx} ({(t_idx + 1)*rx_per_tx} records processed)...")

    # --- SERIALIZATION ---
    df = pd.DataFrame(master_records)
    save_path = os.path.join(save_dir, "raw_data/mirpur_dohs_10k_dataset.csv")
    df.to_csv(save_path, index=False)

    print(f"\n☒ MASSIVE RESEARCH DATASET COMPLETE!")
    print(f"Total Samples Generated: {len(df)}")
    print(f"File Saved to: {save_path}")

# Execute the simulation
generate_massive_dataset(scene, STEP_2_DIR)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# SETUP: Workspace and Styling
# ==========================================
DATA_PATH = '/content/drive/MyDrive/nlos_gnn_shared/step_2_massive_research/raw_data/mirpur_dohs_10k_dataset.csv'
SAVE_DIR = '/content/drive/MyDrive/nlos_gnn_shared/renders/ieee_assets'
os.makedirs(SAVE_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16})

def generate_ieee_assets(csv_path, actuals, predictions, save_dir):
    print("🎨 Initializing IEEE Asset Generator...")
    df = pd.read_csv(csv_path)

    # Asset 1: Distance Decay
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x='distance_2d', y='path_loss_db', data=df, alpha=0.3, color='indigo', s=20)
    plt.title('NLoS Path Loss Decay (28 GHz)')
    plt.savefig(os.path.join(save_dir, 'fig_distance_decay.png'), dpi=300)
    plt.close()

    # Asset 2: Distribution
    plt.figure(figsize=(8, 6))
    sns.histplot(df['path_loss_db'], bins=50, kde=True, color='teal')
    plt.title('Distribution of Path Loss')
    plt.savefig(os.path.join(save_dir, 'fig_loss_distribution.png'), dpi=300)
    plt.close()

    # Asset 3: Error CDF
    absolute_errors = np.abs(actuals - predictions)
    sorted_errors = np.sort(absolute_errors)
    cumulative_prob = np.arange(1, len(sorted_errors) + 1) / len(sorted_errors)

    plt.figure(figsize=(8, 6))
    plt.plot(sorted_errors, cumulative_prob, color='darkred', linewidth=2, label='GNN Model')
    percentile_90 = np.percentile(absolute_errors, 90)
    plt.axhline(y=0.90, color='gray', linestyle='--')
    plt.text(percentile_90 + 0.2, 0.85, f'90% < {percentile_90:.2f} dB')
    plt.title('CDF of Prediction Error')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'fig_error_cdf.png'), dpi=300)
    plt.close()
    print(f"🚀 Phase 1 Complete! Assets saved to {save_dir}")

try:
    generate_ieee_assets(DATA_PATH, actual_db, predicted_db, SAVE_DIR)
except NameError:
    print("⚠️ Training results not found. Generating assets using dataset sample for demonstration...")
    temp_df = pd.read_csv(DATA_PATH).head(1000)
    mock_actuals = temp_df['path_loss_db'].values
    mock_preds = mock_actuals + np.random.normal(0, 2.5, size=len(mock_actuals))
    generate_ieee_assets(DATA_PATH, mock_actuals, mock_preds, SAVE_DIR)

In [1]:
# File: /content/drive/MyDrive/nlos_gnn_shared/notebooks/04_draw_diagrams.py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

SAVE_DIR = '/content/drive/MyDrive/nlos_gnn_shared/renders/ieee_assets'
os.makedirs(SAVE_DIR, exist_ok=True)

def draw_architecture_diagram(save_dir):
    print("🎨 Drawing System Architecture Diagram...")
    fig, ax = plt.subplots(figsize=(12, 3), dpi=300)
    ax.axis('off')

    # Define Block properties
    box_style = "round,pad=0.5"
    props = dict(boxstyle=box_style, facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
    arrow_props = dict(facecolor='black', edgecolor='black', arrowstyle='-|>', lw=2, mutation_scale=15)

    # 1. OSM Block
    ax.text(0.1, 0.5, "1. GIS Data\n(OpenStreetMap\n1,745 Buildings)", ha="center", va="center", size=11, bbox=props)
    # 2. Sionna Block
    ax.text(0.35, 0.5, "2. Digital Twin\n(Sionna 28 GHz\nRay-Tracing)", ha="center", va="center", size=11, bbox=props)
    # 3. Graph Block
    ax.text(0.6, 0.5, "3. Topology Builder\n(K-Nearest Neighbors\nGraph Construction)", ha="center", va="center", size=11, bbox=props)
    # 4. GAT Block
    ax.text(0.85, 0.5, "4. Deep GAT\n(Attention +\nResidual Connections)", ha="center", va="center", size=11, bbox=props)

    # Arrows
    ax.annotate('', xy=(0.23, 0.5), xytext=(0.18, 0.5), arrowprops=arrow_props)
    ax.annotate('', xy=(0.48, 0.5), xytext=(0.44, 0.5), arrowprops=arrow_props)
    ax.annotate('', xy=(0.74, 0.5), xytext=(0.71, 0.5), arrowprops=arrow_props)

    # Path Loss Output Arrow
    ax.annotate('', xy=(0.98, 0.5), xytext=(0.94, 0.5), arrowprops=arrow_props)
    ax.text(0.99, 0.5, "Predicted\nPath Loss (dB)", ha="left", va="center", size=11, fontweight='bold', color='#B71C1C')

    plt.tight_layout()
    save_path = os.path.join(save_dir, 'fig_architecture.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f"✅ Architecture Diagram Saved: {save_path}")

def draw_gat_math_diagram(save_dir):
    print("📐 Drawing GAT Mathematical Diagram...")
    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
    ax.axis('off')

    # Central Node (Target)
    circle_i = patches.Circle((0.5, 0.5), 0.08, facecolor='#FFCDD2', edgecolor='#C62828', lw=2, zorder=3)
    ax.add_patch(circle_i)
    ax.text(0.5, 0.5, "Node $i$\n(Target)", ha="center", va="center", fontsize=12, fontweight='bold', zorder=4)

    # Neighbor Nodes
    neighbors = [
        ((0.2, 0.8), "Neighbor $j_1$"),
        ((0.8, 0.8), "Neighbor $j_2$"),
        ((0.2, 0.2), "Neighbor $j_3$"),
        ((0.8, 0.2), "Neighbor $j_4$")
    ]

    for (nx, ny), label in neighbors:
        circle_n = patches.Circle((nx, ny), 0.06, facecolor='#C8E6C9', edgecolor='#2E7D32', lw=2, zorder=3)
        ax.add_patch(circle_n)
        ax.text(nx, ny, label, ha="center", va="center", fontsize=9, zorder=4)

        # Draw Arrow to Target
        ax.annotate('', xy=(0.5, 0.5), xytext=(nx, ny),
                    arrowprops=dict(facecolor='#424242', edgecolor='#424242', width=1, headwidth=8, shrink=0.15), zorder=2)

    # Math Annotations
    ax.text(0.35, 0.65, "$\\alpha_{ij_1}$", fontsize=14, color='#1565C0', fontweight='bold')
    ax.text(0.65, 0.65, "$\\alpha_{ij_2}$", fontsize=14, color='#1565C0', fontweight='bold')

    # Main Equation at the bottom
    eq_text = r"$\mathbf{h}_i^{(l+1)} = \sigma \left( \sum_{j \in \mathcal{N}(i)} \alpha_{ij} \mathbf{W} \mathbf{h}_j^{(l)} \right) + \mathbf{h}_i^{(l)}$"
    ax.text(0.5, 0.0, eq_text, ha="center", va="center", fontsize=14, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))

    plt.tight_layout()
    save_path = os.path.join(save_dir, 'fig_gat_math.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f"✅ GAT Math Diagram Saved: {save_path}")

# Execute Functions
draw_architecture_diagram(SAVE_DIR)
draw_gat_math_diagram(SAVE_DIR)
print("\n🎉 All missing assets generated successfully!")

🎨 Drawing System Architecture Diagram...
✅ Architecture Diagram Saved: /content/drive/MyDrive/nlos_gnn_shared/renders/ieee_assets/fig_architecture.png
📐 Drawing GAT Mathematical Diagram...
✅ GAT Math Diagram Saved: /content/drive/MyDrive/nlos_gnn_shared/renders/ieee_assets/fig_gat_math.png

🎉 All missing assets generated successfully!
